# KTMD Kin2 / Su corrected montage: lightweight full rerun

この軽量版は、以前のノートブックに埋め込まれていた約8 MBのZIPをBase64文字列として保持しません。  
そのため、Colabで高速に開け、通常のPythonコードとして読めます。

## 実行方法

1. このノートブックと `KTMD_corrected_fullrerun_asset_v4_submission.zip` をダウンロードします。
2. ノートブックをGoogle Colabで開きます。
3. **ランタイム → すべてのセルを実行** を選びます。
4. Google Driveアクセスを許可します。
5. asset ZIPがDriveにまだない場合だけ、ファイル選択画面が1回開くので、上記ZIPを選びます。

選択されたasset ZIPは自動的に
`MyDrive/Global Workspace Analysis/KTMD_corrected_fullrerun_asset_v4_submission.zip`
へ保存されるため、2回目以降は選択不要です。

解析の出力先、raw dataの探索、6日分のSHA-256監査、completion auditは従来版と同一です。


In [ ]:
from google.colab import drive
import subprocess, sys

drive.mount("/content/drive", force_remount=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "networkx", "gdown", "scikit-learn"],
    check=True,
)
print("Drive mounted and dependencies installed.")


In [ ]:
from pathlib import Path
import hashlib
import shutil
import sys
import zipfile
from google.colab import files

ASSET_NAME = "KTMD_corrected_fullrerun_asset_v4_submission.zip"
EXPECTED_ASSET_SHA256 = "861bb97c0fe44f05506ee42bb3e027956d399bb7dd3ded1ba94f95e48cbf4969"
DRIVE_ASSET = Path(
    "/content/drive/MyDrive/Global Workspace Analysis/"
    "KTMD_corrected_fullrerun_asset_v4_submission.zip"
)
LOCAL_ASSET = Path("/content") / ASSET_NAME
ASSET_DIR = Path("/content/KTMD_corrected_fullrerun_asset_v4_submission")

# Prefer the persistent Drive copy. A directly uploaded Colab file is the
# second choice. Only when neither exists is a file picker shown.
if DRIVE_ASSET.exists():
    asset_zip = DRIVE_ASSET
    print("Using persistent Drive asset:", asset_zip)
elif LOCAL_ASSET.exists():
    asset_zip = LOCAL_ASSET
    print("Using asset already uploaded to this Colab runtime:", asset_zip)
else:
    print("Select the sidecar file:", ASSET_NAME)
    uploaded = files.upload()
    if ASSET_NAME not in uploaded:
        raise RuntimeError(
            f"Please select {ASSET_NAME}. Uploaded files: {list(uploaded)}"
        )
    LOCAL_ASSET.write_bytes(uploaded[ASSET_NAME])
    asset_zip = LOCAL_ASSET

observed_sha = hashlib.sha256(asset_zip.read_bytes()).hexdigest()
if observed_sha != EXPECTED_ASSET_SHA256:
    raise RuntimeError(
        "Analysis asset SHA-256 mismatch. "
        f"Expected {EXPECTED_ASSET_SHA256}, observed {observed_sha}"
    )

# Persist an uploaded asset in Drive so future reruns need no upload prompt.
if asset_zip != DRIVE_ASSET:
    DRIVE_ASSET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(asset_zip, DRIVE_ASSET)
    print("Saved a persistent copy to:", DRIVE_ASSET)

if ASSET_DIR.exists():
    shutil.rmtree(ASSET_DIR)
ASSET_DIR.mkdir(parents=True)
with zipfile.ZipFile(asset_zip) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f"Asset ZIP is corrupt at {bad}")
    zf.extractall(ASSET_DIR)

sys.path.insert(0, str(ASSET_DIR))
print("Submission-ready assets extracted:", ASSET_DIR)
print("Asset SHA-256:", observed_sha)


In [ ]:
from pathlib import Path
import shutil
import gdown
from run_full_corrected_pipeline import TARGETS, load_expected, find_manifest_and_parts

PUBLIC_SPLIT_FOLDER = "https://drive.google.com/drive/folders/1j8BZzWSOpinykXxf7W63HPxgJquiHgEU?usp=drive_link"
EXPECTED = load_expected(ASSET_DIR)

CANDIDATES = [
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Split80MiB"),
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Downloads"),
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Downloads/NeuroTycho_Monkey_ECoG_Split80MiB"),
]

def recognized_targets(root: Path) -> int:
    if not root.exists():
        return 0
    count = 0
    for animal, date in TARGETS:
        name = f"{date}KTMD_Anesthesia+and+Sleep_{animal}_Toru+Yanagawa_mat_ECoG128.zip"
        try:
            find_manifest_and_parts(root, animal, date, EXPECTED[name])
            count += 1
        except Exception:
            pass
    return count

SPLIT_ROOT = None
for candidate in CANDIDATES:
    n = recognized_targets(candidate)
    print(f"Input audit: {candidate} -> {n}/6 target archives")
    if n == 6:
        SPLIT_ROOT = candidate
        break

if SPLIT_ROOT is None:
    PUBLIC_CACHE = Path("/content/ktmd_shared_parts")
    if PUBLIC_CACHE.exists() and recognized_targets(PUBLIC_CACHE) < 6:
        shutil.rmtree(PUBLIC_CACHE)
    PUBLIC_CACHE.mkdir(parents=True, exist_ok=True)
    print("The six complete split archives were not found in MyDrive; downloading the public shared folder...")
    gdown.download_folder(
        url=PUBLIC_SPLIT_FOLDER,
        output=str(PUBLIC_CACHE),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )
    n = recognized_targets(PUBLIC_CACHE)
    print(f"Public-folder audit: {n}/6 target archives")
    if n != 6:
        raise RuntimeError(
            "The public folder download did not expose all six complete archive manifests/parts. "
            "Check that the shared folder permission remains 'Anyone with the link'."
        )
    SPLIT_ROOT = PUBLIC_CACHE

print("Verified split root:", SPLIT_ROOT)


In [ ]:
from pathlib import Path
import subprocess, sys, time

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Global Workspace Analysis/"
    "KTMD_Kin2_Su_CorrectedMontage_FullRerun_SubmissionReady_20260816"
)
WORK_ROOT = Path("/content/ktmd_corrected_fullrerun_work")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_ROOT / "FULL_RERUN_CONSOLE.log"

cmd = [
    sys.executable, "-u", str(ASSET_DIR / "run_full_corrected_pipeline.py"),
    "--split-root", str(SPLIT_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--assets", str(ASSET_DIR),
    "--work-root", str(WORK_ROOT),
]
print("Running:", " ".join(cmd))
print("Output:", OUTPUT_ROOT)
started = time.time()
with LOG_PATH.open("a", encoding="utf-8") as log:
    log.write("\n\n=== NEW RUN ===\n")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print(f"Full rerun finished in {(time.time()-started)/60:.1f} min")


In [ ]:
from pathlib import Path
import json
import pandas as pd
import subprocess, sys

required = {
    "completion marker": OUTPUT_ROOT / "ALL_ANALYSES_COMPLETE.ok",
    "raw provenance marker": OUTPUT_ROOT / "REAL_RAW_DATA_PROVENANCE_VERIFIED.ok",
    "raw provenance table": OUTPUT_ROOT / "tables" / "RAW_ARCHIVE_PROVENANCE_ALL_SIX_DAYS.csv",
    "completion audit": OUTPUT_ROOT / "COMPLETION_AUDIT.json",
    "final summary": OUTPUT_ROOT / "FINAL_SUMMARY.json",
    "updated LODO table": OUTPUT_ROOT / "tables/updated_hierarchical_lodo_deep_effects.csv",
    "updated subject maps": OUTPUT_ROOT / "figures/figure_updated_subject_specific_official_maps.png",
    "updated cross-day figure": OUTPUT_ROOT / "figures/figure_updated_crossday_transfer.png",
    "writing-thread JSON": OUTPUT_ROOT / "manuscript_materials/RESULTS_FOR_WRITING_THREAD.json",
    "Results patch": OUTPUT_ROOT / "manuscript_materials/MANUSCRIPT_RESULTS_PATCH.md",
}
print("FINAL COMPLETION AUDIT")
for label, path in required.items():
    print("[OK]" if path.exists() else "[MISSING]", label, path)
if not all(path.exists() for path in required.values()):
    raise RuntimeError("One or more required manuscript outputs are missing")

audit = json.loads((OUTPUT_ROOT / "COMPLETION_AUDIT.json").read_text())
if not audit.get("all_required_complete", False):
    raise RuntimeError(f"Strict completion audit failed: {audit}")
print(json.dumps(audit, indent=2))
print("\nUpdated animal-balanced LODO results:")
display(pd.read_csv(OUTPUT_ROOT / "tables/updated_hierarchical_lodo_deep_effects.csv"))

DERIVED_ZIP = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_DERIVED_RESULTS.zip")
subprocess.run([sys.executable,str(ASSET_DIR/"strict_postrun_verifier.py"),str(OUTPUT_ROOT)],check=True)
print("\nALL SIX CORRECTED RERUNS COMPLETE — RAW PROVENANCE LOCKED")
print("Result folder:", OUTPUT_ROOT)
print("Compact manuscript handoff ZIP:", DERIVED_ZIP)
